In [ ]:
import json
from collections import Counter

file_path = "manual_baseline.jsonl"

conference_counter = Counter()

with open(file_path, "r", encoding="utf-8") as f:
    for line in f:
        data = json.loads(line.strip())
        conf = data.get("conference")
        if conf:
            conference_counter[conf] += 1

# Print results
print("Conference Distribution:\n")
for conf, count in conference_counter.items():
    print(f"{conf}: {count}")

# Optional: percentage
total = sum(conference_counter.values())
print("\nWith Percentage:\n")
for conf, count in conference_counter.items():
    print(f"{conf}: {count} ({count/total:.2%})")

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter

file_path = "manual_baseline.jsonl"

rows = []

with open(file_path, "r", encoding="utf-8") as f:
    for line in f:
        paper = json.loads(line.strip())

        title = paper.get("title")
        paper_id = paper.get("paper_id")
        conf = paper.get("conference")
        year = paper.get("year")

        dataset_field = paper.get("dataset_name")

        if isinstance(dataset_field, str):
            try:
                dataset_field = json.loads(dataset_field)
            except json.JSONDecodeError:
                continue

        datasets = dataset_field.get("datasets", []) if isinstance(dataset_field, dict) else []

        for d in datasets:
            rows.append({
                "title": title,
                "paper_id": paper_id,
                "conference": conf,
                "year": int(year) if year else None,
                "dataset_name": d.get("dataset_name"),
                "action": d.get("action")
            })

df = pd.DataFrame(rows)

# normalize action names
df["action"] = df["action"].str.lower().str.strip()

print("\nOverall Created vs Used Dataset Instances")
print(df["action"].value_counts())

print("\nCreated vs Used by Conference")
conf_dist = pd.crosstab(df["conference"], df["action"])
print(conf_dist)

print("\nCreated vs Used by Year")
year_dist = pd.crosstab(df["year"], df["action"]).sort_index()
print(year_dist)


# -----------------------------
# Plot 1: Overall distribution
# -----------------------------
plt.figure(figsize=(6, 4))
df["action"].value_counts().plot(kind="bar")
plt.title("Overall Dataset Action Distribution")
plt.xlabel("Action Type")
plt.ylabel("Number of Dataset Instances")
plt.tight_layout()
plt.show()


# -----------------------------
# Plot 2: Per-conference distribution
# -----------------------------
plt.figure(figsize=(8, 5))
conf_dist.plot(kind="bar", figsize=(8, 5))
plt.title("Dataset Action Type by Conference")
plt.xlabel("Conference")
plt.ylabel("Number of Dataset Instances")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


# -----------------------------
# Plot 3: Temporal line graph
# -----------------------------
plt.figure(figsize=(9, 5))

for action in year_dist.columns:
    plt.plot(year_dist.index, year_dist[action], marker="o", label=action)

plt.title("Dataset Usage and Creation Over Time")
plt.xlabel("Year")
plt.ylabel("Number of Dataset Instances")
plt.legend(title="Action Type")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt

file_path = "manual_baseline.jsonl"

rows = []

with open(file_path, "r", encoding="utf-8") as f:
    for line in f:
        paper = json.loads(line.strip())
        rows.append({
            "conference": paper.get("conference"),
            "year": paper.get("year"),
            "paper_type": paper.get("paper_type")
        })

df = pd.DataFrame(rows)

# Raw counts by conference
conf_dist = pd.crosstab(df["conference"], df["paper_type"])

print("Paper-level action distribution by conference:")
print(conf_dist)

# Percentages by conference
conf_percent = conf_dist.div(conf_dist.sum(axis=1), axis=0) * 100

print("\nPercentage distribution by conference:")
print(conf_percent.round(2))

# Save tables
conf_dist.to_csv("paper_type_by_conference_counts.csv")
conf_percent.to_csv("paper_type_by_conference_percent.csv")

# Stacked bar chart: counts
conf_dist.plot(kind="bar", stacked=True, figsize=(8, 5))
plt.title("Paper-Level Dataset Usage and Creation by Conference")
plt.xlabel("Conference")
plt.ylabel("Number of Papers")
plt.xticks(rotation=0)
plt.legend(title="Paper Type")
plt.tight_layout()
plt.savefig("paper_type_by_conference_counts.png", dpi=300)
plt.show()

# Stacked bar chart: percentages
conf_percent.plot(kind="bar", stacked=True, figsize=(8, 5))
plt.title("Paper-Level Dataset Usage and Creation by Conference")
plt.xlabel("Conference")
plt.ylabel("Percentage of Papers")
plt.xticks(rotation=0)
plt.legend(title="Paper Type")
plt.tight_layout()
plt.savefig("paper_type_by_conference_percent.png", dpi=300)
plt.show()

RQ1

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt


data = {
    "conference": ["CCS", "NDSS", "S&P", "USS"],
    "used_only": [45.55, 43.09, 48.40, 39.69],
    "both": [30.60, 30.18, 30.22, 34.88],
    "created_only": [23.84, 26.73, 21.38, 25.44],
}

df = pd.DataFrame(data).set_index("conference")

# Better display order
df = df[["used_only", "both", "created_only"]]

ax = df.plot(
    kind="barh",
    stacked=True,
    figsize=(7, 3.2),
    edgecolor="white"
)

ax.set_xlim(0, 100)
ax.set_xlabel("Percentage of Papers")
ax.set_ylabel("")
ax.set_title("Paper-Level Dataset Usage and Creation by Conference")

ax.legend(
    ["Used only", "Both", "Created only"],
    title="Paper Type",
    loc="upper center",
    bbox_to_anchor=(0.5, 1.25),
    ncol=3,
    frameon=True
)

ax.xaxis.set_major_formatter(lambda x, pos: f"{int(x)}%")
ax.grid(axis="x", linestyle="--", alpha=0.5)

plt.tight_layout()
plt.savefig("paper_type_by_conference_percent.pdf", bbox_inches="tight")
plt.savefig("paper_type_by_conference_percent.png", dpi=300, bbox_inches="tight")
plt.show()

RQ2

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt

file_path = "manual_baseline.jsonl"

rows = []

with open(file_path, "r", encoding="utf-8") as f:
    for line in f:
        paper = json.loads(line.strip())
        dataset_field = paper.get("dataset_name")

        if isinstance(dataset_field, str):
            dataset_field = json.loads(dataset_field)

        for d in dataset_field.get("datasets", []):
            rows.append({
                "conference": paper.get("conference"),
                "year": paper.get("year"),
                "action": d.get("action", "").lower().strip()
            })

df = pd.DataFrame(rows)

# -----------------------------
# Dataset-level per conference
# -----------------------------
conf_dist = pd.crosstab(df["conference"], df["action"])

# Convert to percentage (better for comparison)
conf_percent = conf_dist.div(conf_dist.sum(axis=1), axis=0) * 100

print("\nDataset-level distribution by conference:")
print(conf_percent.round(2))

# Save
conf_percent.to_csv("dataset_level_by_conference_percent.csv")

# -----------------------------
# Plot (STACKED BAR - BEST)
# -----------------------------
conf_percent = conf_percent[["used", "created"]]  # ensure order

conf_percent.plot(
    kind="bar",
    stacked=True,
    figsize=(7,5)
)

plt.title("Dataset-Level Usage and Creation by Conference")
plt.xlabel("Conference")
plt.ylabel("Percentage of Dataset Instances")
plt.xticks(rotation=0)
plt.legend(title="Action Type")
plt.tight_layout()
plt.savefig("dataset_level_by_conference.png", dpi=300)
plt.show()

temporal trend

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt

file_path = "manual_baseline.jsonl"

rows = []

with open(file_path, "r", encoding="utf-8") as f:
    for line in f:
        paper = json.loads(line.strip())
        rows.append({
            "year": int(paper.get("year")),
            "paper_type": paper.get("paper_type")
        })

df = pd.DataFrame(rows)

# -----------------------------
# Group by year
# -----------------------------
year_dist = pd.crosstab(df["year"], df["paper_type"])

# Convert to percentage (important!)
year_percent = year_dist.div(year_dist.sum(axis=1), axis=0) * 100

# Sort by year
year_percent = year_percent.sort_index()

print("\nTemporal paper-level distribution (%):")
print(year_percent.round(2))

# Save
year_percent.to_csv("paper_level_temporal_percent.csv")

# -----------------------------
# Plot (LINE GRAPH)
# -----------------------------
plt.figure(figsize=(8,5))

for col in ["used_only", "both", "created_only"]:
    if col in year_percent.columns:
        plt.plot(
            year_percent.index,
            year_percent[col],
            marker="o",
            label=col
        )

plt.title("Temporal Trends in Dataset Usage (Paper-Level)")
plt.xlabel("Year")
plt.ylabel("Percentage of Papers")
plt.legend(title="Paper Type")
plt.grid(True, linestyle="--", alpha=0.5)

plt.tight_layout()
plt.savefig("paper_level_temporal.png", dpi=300)
plt.show()

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt

file_path = "manual_baseline.jsonl"

rows = []

with open(file_path, "r", encoding="utf-8") as f:
    for line in f:
        paper = json.loads(line.strip())
        rows.append({
            "conference": paper.get("conference"),
            "year": int(paper.get("year")),
            "paper_type": paper.get("paper_type")
        })

df = pd.DataFrame(rows)

# keep valid rows
df = df.dropna(subset=["conference", "year", "paper_type"])

# conference-year-paper_type percentage
trend = (
    df.groupby(["conference", "year", "paper_type"])
      .size()
      .reset_index(name="count")
)

trend["total_per_conf_year"] = trend.groupby(["conference", "year"])["count"].transform("sum")
trend["percent"] = trend["count"] / trend["total_per_conf_year"] * 100

# save table
trend.to_csv("paper_level_temporal_by_conference.csv", index=False)

print(trend)

# plot one figure per paper_type across conferences
paper_types = ["used_only", "both", "created_only"]

for ptype in paper_types:
    plt.figure(figsize=(8, 5))

    for conf in sorted(df["conference"].unique()):
        temp = trend[(trend["conference"] == conf) & (trend["paper_type"] == ptype)]
        plt.plot(
            temp["year"],
            temp["percent"],
            marker="o",
            label=conf
        )

    plt.title(f"Temporal Trend of {ptype.replace('_', ' ').title()} Papers by Conference")
    plt.xlabel("Year")
    plt.ylabel("Percentage of Papers")
    plt.ylim(0, 70)
    plt.grid(True, linestyle="--", alpha=0.5)
    plt.legend(title="Conference")
    plt.tight_layout()
    plt.savefig(f"temporal_{ptype}_by_conference.png", dpi=300)
    plt.show()

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt

file_path = "manual_baseline.jsonl"

rows = []

with open(file_path, "r", encoding="utf-8") as f:
    for line in f:
        paper = json.loads(line.strip())
        rows.append({
            "conference": paper.get("conference"),
            "year": int(paper.get("year")),
            "paper_type": paper.get("paper_type")
        })

df = pd.DataFrame(rows).dropna(subset=["conference", "year", "paper_type"])

trend = (
    df.groupby(["conference", "year", "paper_type"])
      .size()
      .reset_index(name="count")
)

trend["total"] = trend.groupby(["conference", "year"])["count"].transform("sum")
trend["percent"] = trend["count"] / trend["total"] * 100

# Save full long-format table
trend.to_csv("temporal_paper_type_by_conference_long.csv", index=False)

paper_types = ["used_only", "both", "created_only"]

print("\nTemporal paper-level distribution by conference (%):\n")

for ptype in paper_types:
    pivot = (
        trend[trend["paper_type"] == ptype]
        .pivot(index="year", columns="conference", values="percent")
        .sort_index()
        .round(2)
    )
    
    print(f"\n=== {ptype} ===")
    print(pivot)
    
    pivot.to_csv(f"temporal_{ptype}_by_conference_percent.csv")

# Plot
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)

for i, ptype in enumerate(paper_types):
    ax = axes[i]

    for conf in sorted(df["conference"].unique()):
        temp = trend[
            (trend["conference"] == conf) &
            (trend["paper_type"] == ptype)
        ].sort_values("year")

        ax.plot(
            temp["year"],
            temp["percent"],
            marker="o",
            label=conf
        )

    ax.set_title(ptype.replace("_", " ").title())
    ax.set_xlabel("Year")
    ax.grid(True, linestyle="--", alpha=0.5)

    if i == 0:
        ax.set_ylabel("Percentage")

handles, labels = axes[-1].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", ncol=4)

plt.tight_layout(rect=[0, 0, 1, 0.88])

plt.savefig("temporal_all_in_one.png", dpi=300, bbox_inches="tight")
plt.savefig("temporal_all_in_one.pdf", bbox_inches="tight")

plt.show()

RQ2-B Reuse (Frequency & Statistical Analysis)

In [ ]:
import json
import pandas as pd

file_path = "manual_baseline.jsonl"

def safe_json_loads(x):
    if isinstance(x, (dict, list)):
        return x
    if isinstance(x, str):
        try:
            return json.loads(x)
        except json.JSONDecodeError:
            return None
    return None

def is_valid_dataset(name):
    if not name:
        return False
    return True  # no dataset-specific filtering

rows = []
bad_lines = 0

with open(file_path, "r", encoding="utf-8") as f:
    for i, line in enumerate(f, start=1):
        line = line.strip()
        if not line:
            continue

        try:
            paper = json.loads(line)
        except json.JSONDecodeError:
            bad_lines += 1
            print(f"Skipping invalid JSON at line {i}")
            continue

        dataset_field = safe_json_loads(paper.get("dataset_name"))
        category_field = safe_json_loads(paper.get("dataset_categories"))

        if not isinstance(dataset_field, dict):
            continue

        # Build lookup: dataset_name -> category info
        category_lookup = {}
        if isinstance(category_field, dict):
            for c in category_field.get("datasets", []):
                cname = str(c.get("dataset_name", "")).strip()
                if cname:
                    category_lookup[cname] = {
                        "category": c.get("category"),
                        "subcategory": c.get("subcategory")
                    }

        for d in dataset_field.get("datasets", []):
            if str(d.get("action", "")).lower().strip() == "used":
                name = str(d.get("dataset_name", "")).strip()

                if is_valid_dataset(name):
                    cat_info = category_lookup.get(name, {})

                    rows.append({
                        "dataset_name": name,
                        "conference": paper.get("conference"),
                        "year": int(paper.get("year")),
                        "category": cat_info.get("category", "unknown"),
                        "subcategory": cat_info.get("subcategory", "unknown")
                    })

df = pd.DataFrame(rows)

print(f"\nSkipped {bad_lines} invalid lines")
print(f"Total dataset instances: {len(df)}")

# Dataset frequency
freq = (
    df.groupby(["dataset_name", "category", "subcategory"])
      .size()
      .reset_index(name="usage_count")
      .sort_values(by="usage_count", ascending=False)
)

freq.to_csv("dataset_frequency.csv", index=False)

print("\nTop datasets:\n")
print(freq.head(40))

# Top-k contribution %
total_usage = freq["usage_count"].sum()

for k in [5, 10, 15, 20]:
    topk_count = freq.head(k)["usage_count"].sum()
    percent = 100 * topk_count / total_usage if total_usage else 0
    print(f"Top {k}: {topk_count} instances ({percent:.2f}%)")

# Category distribution
cat_dist = (
    df.groupby("category")
      .size()
      .reset_index(name="count")
      .sort_values(by="count", ascending=False)
)

cat_dist["percent"] = cat_dist["count"] / cat_dist["count"].sum() * 100
cat_dist.to_csv("category_distribution.csv", index=False)

print("\nCategory distribution:\n")
print(cat_dist.round(2))

# Subcategory distribution
subcat_dist = (
    df.groupby(["category", "subcategory"])
      .size()
      .reset_index(name="count")
      .sort_values(by="count", ascending=False)
)

subcat_dist["percent"] = subcat_dist["count"] / subcat_dist["count"].sum() * 100
subcat_dist.to_csv("subcategory_distribution.csv", index=False)

print("\nTop subcategories:\n")
print(subcat_dist.head(40).round(2))

# General-purpose share (top datasets)
top_k = freq.head(40).copy()
top_k_total = top_k["usage_count"].sum()

mask = top_k["category"].str.lower().eq("general_purpose_data_modalities")
gp_count = top_k.loc[mask, "usage_count"].sum()

percent = 100 * gp_count / top_k_total if top_k_total else 0

print("\nWithin top datasets:")
print(f"General-purpose usage: {gp_count}/{top_k_total} ({percent:.2f}%)")

top_k.to_csv("top_datasets.csv", index=False)

statistical analysis of existing datasets


In [ ]:
import json
import pandas as pd
from scipy.stats import chi2_contingency, spearmanr

file_path = "manual_baseline.jsonl"

def safe_json_loads(x):
    if isinstance(x, (dict, list)):
        return x
    if isinstance(x, str):
        try:
            return json.loads(x)
        except json.JSONDecodeError:
            return None
    return None

def parse_list_field(x):
    parsed = safe_json_loads(x)

    if isinstance(parsed, list):
        return [str(v).strip() for v in parsed if str(v).strip()]

    if isinstance(x, str):
        return [v.strip() for v in x.split(",") if v.strip()]

    return ["unknown"]

def is_valid_dataset(name):
    if not name:
        return False
    return True  # no dataset-specific filtering

used_rows = []
used_domain_rows = []
bad_lines = 0

with open(file_path, "r", encoding="utf-8") as f:
    for i, line in enumerate(f, start=1):
        line = line.strip()
        if not line:
            continue

        try:
            paper = json.loads(line)
        except json.JSONDecodeError:
            bad_lines += 1
            print(f"Skipping invalid JSON at line {i}")
            continue

        dataset_field = safe_json_loads(paper.get("dataset_name"))
        category_field = safe_json_loads(paper.get("dataset_categories"))

        if not isinstance(dataset_field, dict):
            continue

        domains = parse_list_field(paper.get("high_level_domains"))
        subdomains = parse_list_field(paper.get("subdomains"))

        max_len = max(len(domains), len(subdomains))

        while len(domains) < max_len:
            domains.append("unknown")
        while len(subdomains) < max_len:
            subdomains.append("unknown")

        domain_pairs = list(zip(domains, subdomains))

        category_lookup = {}
        if isinstance(category_field, dict):
            for c in category_field.get("datasets", []):
                cname = str(c.get("dataset_name", "")).strip()
                if cname:
                    category_lookup[cname] = {
                        "category": c.get("category", "unknown"),
                        "dataset_subcategory": c.get("subcategory", "unknown")
                    }

        for d in dataset_field.get("datasets", []):
            action = str(d.get("action", "")).lower().strip()

            if action != "used":
                continue

            name = str(d.get("dataset_name", "")).strip()

            if not is_valid_dataset(name):
                continue

            category_info = category_lookup.get(name, {})

            base_row = {
                "dataset_name": name,
                "action": action,
                "conference": paper.get("conference"),
                "year": int(paper.get("year")),
                "category": category_info.get("category", "unknown"),
                "dataset_subcategory": category_info.get("dataset_subcategory", "unknown")
            }

            used_rows.append(base_row)

            for domain, subdomain in domain_pairs:
                domain_row = base_row.copy()
                domain_row["domain"] = domain
                domain_row["subdomain"] = subdomain
                used_domain_rows.append(domain_row)

df_used = pd.DataFrame(used_rows)
df_used_domain = pd.DataFrame(used_domain_rows)

print("\nDataset usage summary")
print("Skipped invalid JSON lines:", bad_lines)
print("Dataset instances:", len(df_used))
print("Dataset-domain instances:", len(df_used_domain))

freq = (
    df_used.groupby(["dataset_name", "category", "dataset_subcategory"])
    .size()
    .reset_index(name="usage_count")
    .sort_values("usage_count", ascending=False)
)

freq.to_csv("dataset_frequency.csv", index=False)

cat_usage = (
    df_used.groupby("category")
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)

cat_usage["percent"] = cat_usage["count"] / cat_usage["count"].sum() * 100
cat_usage.to_csv("category_distribution.csv", index=False)

domain_usage = (
    df_used_domain.groupby("domain")
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)

domain_usage["percent"] = domain_usage["count"] / domain_usage["count"].sum() * 100
domain_usage.to_csv("domain_distribution.csv", index=False)

subdomain_usage = (
    df_used_domain.groupby("subdomain")
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)

subdomain_usage["percent"] = subdomain_usage["count"] / subdomain_usage["count"].sum() * 100
subdomain_usage.to_csv("subdomain_distribution.csv", index=False)

year_counts = (
    df_used.groupby("year")
    .size()
    .reset_index(name="dataset_count")
    .sort_values("year")
)

year_counts.to_csv("dataset_counts_by_year.csv", index=False)

table_cat_conf = pd.crosstab(df_used["category"], df_used["conference"])
chi2_cat_conf, p_cat_conf, dof_cat_conf, _ = chi2_contingency(table_cat_conf)

table_cat_year = pd.crosstab(df_used["category"], df_used["year"])
chi2_cat_year, p_cat_year, dof_cat_year, _ = chi2_contingency(table_cat_year)

rho, p_spearman = spearmanr(
    year_counts["year"],
    year_counts["dataset_count"]
)

stats_summary = pd.DataFrame([
    {
        "variable": "Category vs Conference",
        "test": "Chi-square",
        "statistic": chi2_cat_conf,
        "dof": dof_cat_conf,
        "p_value": p_cat_conf
    },
    {
        "variable": "Category vs Year",
        "test": "Chi-square",
        "statistic": chi2_cat_year,
        "dof": dof_cat_year,
        "p_value": p_cat_year
    },
    {
        "variable": "Year vs Dataset Count",
        "test": "Spearman",
        "statistic": rho,
        "dof": None,
        "p_value": p_spearman
    }
])

stats_summary.to_csv("statistical_analysis.csv", index=False)

print(stats_summary)

In [ ]:
!pip install statsmodels

In [ ]:
import numpy as np

def check_chi_square(table, name):
    from scipy.stats import chi2_contingency
    chi2, p, dof, expected = chi2_contingency(table)
    
    # count cells with expected < 5
    low_counts = (expected < 5).sum()
    total_cells = expected.size
    
    print(f"\n{name}")
    print(f"p-value: {p}")
    print(f"Cells with expected <5: {low_counts}/{total_cells}")
    
    if low_counts / total_cells > 0.2:
        print(" Warning: Chi-square assumption may be violated")
    else:
        print(" Chi-square assumptions OK")

# Example checks
check_chi_square(table_cat_conf, "Category vs Conference")
check_chi_square(table_cat_year, "Category vs Year")
check_chi_square(table_cat_domain, "Category vs Domain")

In [ ]:
import numpy as np

def cramers_v(table):
    chi2, _, _, _ = chi2_contingency(table)
    n = table.to_numpy().sum()
    r, c = table.shape
    return np.sqrt(chi2 / (n * (min(r, c) - 1)))

def bh_correction(pvals):
    pvals = np.array(pvals, dtype=float)
    n = len(pvals)

    sorted_idx = np.argsort(pvals)
    sorted_pvals = pvals[sorted_idx]

    corrected_sorted = np.empty(n)
    prev = 1.0

    for i in range(n - 1, -1, -1):
        rank = i + 1
        corrected_value = sorted_pvals[i] * n / rank
        prev = min(prev, corrected_value)
        corrected_sorted[i] = prev

    corrected = np.empty(n)
    corrected[sorted_idx] = corrected_sorted
    return corrected

# Effect sizes
v_cat_conf = cramers_v(table_cat_conf)
v_cat_year = cramers_v(table_cat_year)
v_cat_domain = cramers_v(table_cat_domain)
v_domain_conf = cramers_v(table_domain_conf)
v_domain_year = cramers_v(table_domain_year)

# Multiple-testing correction (Chi-square only)
chi_pvals = [
    p_cat_conf,
    p_cat_year,
    p_cat_domain,
    p_domain_conf,
    p_domain_year
]

corrected_pvals = bh_correction(chi_pvals)

stats_summary = pd.DataFrame([
    {
        "variable": "Category vs Conference",
        "data_used": "dataset instances",
        "test": "Chi-square",
        "statistic": chi2_cat_conf,
        "dof": dof_cat_conf,
        "p_value": p_cat_conf,
        "p_value_fdr_bh": corrected_pvals[0],
        "cramers_v": v_cat_conf
    },
    {
        "variable": "Category vs Year",
        "data_used": "dataset instances",
        "test": "Chi-square",
        "statistic": chi2_cat_year,
        "dof": dof_cat_year,
        "p_value": p_cat_year,
        "p_value_fdr_bh": corrected_pvals[1],
        "cramers_v": v_cat_year
    },
    {
        "variable": "Year vs Dataset Count",
        "data_used": "dataset instances",
        "test": "Spearman",
        "statistic": rho,
        "dof": None,
        "p_value": p_spearman,
        "p_value_fdr_bh": None,
        "cramers_v": None
    },
    {
        "variable": "Category vs Domain",
        "data_used": "dataset-domain instances",
        "test": "Chi-square",
        "statistic": chi2_cat_domain,
        "dof": dof_cat_domain,
        "p_value": p_cat_domain,
        "p_value_fdr_bh": corrected_pvals[2],
        "cramers_v": v_cat_domain
    },
    {
        "variable": "Domain vs Conference",
        "data_used": "dataset-domain instances",
        "test": "Chi-square",
        "statistic": chi2_domain_conf,
        "dof": dof_domain_conf,
        "p_value": p_domain_conf,
        "p_value_fdr_bh": corrected_pvals[3],
        "cramers_v": v_domain_conf
    },
    {
        "variable": "Domain vs Year",
        "data_used": "dataset-domain instances",
        "test": "Chi-square",
        "statistic": chi2_domain_year,
        "dof": dof_domain_year,
        "p_value": p_domain_year,
        "p_value_fdr_bh": corrected_pvals[4],
        "cramers_v": v_domain_year
    }
])

stats_summary.to_csv("statistical_analysis.csv", index=False)

print(stats_summary)

In [ ]:
import json
import pandas as pd
import numpy as np

file_path = "baseline_with_type.jsonl"

def safe_json_loads(x):
    if isinstance(x, list):
        return x
    if isinstance(x, str):
        try:
            return json.loads(x)
        except json.JSONDecodeError:
            return []
    return []

def parse_domains(x):
    parsed = safe_json_loads(x)

    if isinstance(parsed, list):
        return list(set(str(v).strip() for v in parsed if str(v).strip()))

    if isinstance(x, str):
        return list(set(v.strip() for v in x.split(",") if v.strip()))

    return []

paper_rows = []

with open(file_path, "r", encoding="utf-8") as f:
    for line in f:
        try:
            paper = json.loads(line.strip())
        except json.JSONDecodeError:
            continue

        domains = parse_domains(paper.get("high_level_domains"))

        paper_rows.append({
            "conference": paper.get("conference"),
            "domains": domains
        })

df = pd.DataFrame(paper_rows)

all_domains = sorted(set(d for row in df["domains"] for d in row))

result = []

for conf in sorted(df["conference"].dropna().unique()):
    temp = df[df["conference"] == conf]

    row = {"Conference": conf}
    total_papers = len(temp)
    row["Papers"] = total_papers

    for d in all_domains:
        row[d] = 0

    for domains in temp["domains"]:
        for d in domains:
            row[d] += 1

    domain_counts_per_paper = [len(domains) for domains in temp["domains"]]

    row["AvgDom"] = round(np.mean(domain_counts_per_paper), 2)
    row["P50"] = int(np.percentile(domain_counts_per_paper, 50))
    row["P90"] = int(np.percentile(domain_counts_per_paper, 90))

    result.append(row)

df_table = pd.DataFrame(result)

total_row = {"Conference": "Total"}
total_row["Papers"] = len(df)

for d in all_domains:
    total_row[d] = sum(df_table[d])

total_row["AvgDom"] = "-"
total_row["P50"] = "-"
total_row["P90"] = "-"

df_table = pd.concat([df_table, pd.DataFrame([total_row])], ignore_index=True)

print(df_table)
df_table.to_csv("domain_distribution_table.csv", index=False)

In [ ]:
import json
import pandas as pd
import itertools
import numpy as np
import matplotlib.pyplot as plt

file_path = "manual_baseline.jsonl"

def parse_domains(x):
    if isinstance(x, str):
        try:
            x = json.loads(x)
        except json.JSONDecodeError:
            x = x.split(",")

    if isinstance(x, list):
        return sorted(set(str(d).strip() for d in x if str(d).strip()))

    return []

pairs_data = []

with open(file_path, "r", encoding="utf-8") as f:
    for line in f:
        try:
            paper = json.loads(line.strip())
        except json.JSONDecodeError:
            continue

        year = int(paper.get("year"))
        conference = paper.get("conference")
        domains = parse_domains(paper.get("high_level_domains"))

        if len(domains) < 2:
            continue

        for pair in itertools.combinations(domains, 2):
            pairs_data.append({
                "year": year,
                "conference": conference,
                "domain_pair": f"{pair[0]} -- {pair[1]}"
            })

df_pairs = pd.DataFrame(pairs_data)

df_pairs.to_csv("domain_pairs_by_paper.csv", index=False)

pair_counts = (
    df_pairs.groupby("domain_pair")
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)

pair_counts.to_csv("domain_pair_counts.csv", index=False)

top_pairs = pair_counts.head(5)["domain_pair"].tolist()

print("\nTop domain pairs:")
print(pair_counts.head(5))

trend = (
    df_pairs[df_pairs["domain_pair"].isin(top_pairs)]
    .groupby(["year", "domain_pair"])
    .size()
    .reset_index(name="count")
)

all_years = sorted(df_pairs["year"].unique())
full_index = pd.MultiIndex.from_product(
    [all_years, top_pairs],
    names=["year", "domain_pair"]
)

trend_full = (
    trend.set_index(["year", "domain_pair"])
    .reindex(full_index, fill_value=0)
    .reset_index()
)

trend_full.to_csv("domain_pair_yearly_trend.csv", index=False)

print("\nYearly trend for top domain pairs:")
print(trend_full)

cdf_rows = []

for pair in top_pairs:
    counts = trend_full[trend_full["domain_pair"] == pair]["count"].values
    values_sorted = np.sort(counts)
    cdf = np.arange(1, len(values_sorted) + 1) / len(values_sorted)

    for x, y in zip(values_sorted, cdf):
        cdf_rows.append({
            "domain_pair": pair,
            "yearly_count": x,
            "cdf": y
        })

cdf_df = pd.DataFrame(cdf_rows)
cdf_df.to_csv("domain_pair_cdf_data.csv", index=False)

print("\nCDF data saved:")
print(cdf_df.head(20))

plt.figure(figsize=(7, 4))

for pair in top_pairs:
    temp = cdf_df[cdf_df["domain_pair"] == pair]
    plt.plot(temp["yearly_count"], temp["cdf"], marker="o", label=pair)

plt.xlabel("Yearly Paper Count")
plt.ylabel("CDF")
plt.title("CDF of Domain Pair Co-occurrences")
plt.grid(True, linestyle="--", alpha=0.5)
plt.legend(fontsize=7)
plt.tight_layout()
plt.savefig("domain_pair_cdf.png", dpi=300)
plt.show()

In [ ]:
import json
import itertools
import pandas as pd
import matplotlib.pyplot as plt

file_path = "final_baseline_with_paper_type.normalized.subcategory_fixed.updated.jsonl"

def parse_domains(x):
    if isinstance(x, list):
        return sorted(set(str(d).strip() for d in x if str(d).strip()))

    if isinstance(x, str):
        try:
            parsed = json.loads(x)
            if isinstance(parsed, list):
                return sorted(set(str(d).strip() for d in parsed if str(d).strip()))
        except json.JSONDecodeError:
            return sorted(set(d.strip() for d in x.split(",") if d.strip()))

    return []

rows = []

with open(file_path, "r", encoding="utf-8") as f:
    for line in f:
        try:
            paper = json.loads(line.strip())
        except json.JSONDecodeError:
            continue

        year = int(paper.get("year"))
        domains = parse_domains(paper.get("high_level_domains"))

        if len(domains) < 2:
            continue

        for d1, d2 in itertools.combinations(domains, 2):
            rows.append({
                "year": year,
                "domain_pair": f"{d1} -- {d2}"
            })

df_pairs = pd.DataFrame(rows)

# Top 5 domain pairs overall
pair_counts = (
    df_pairs.groupby("domain_pair")
    .size()
    .reset_index(name="total_count")
    .sort_values("total_count", ascending=False)
)

top5_pairs = pair_counts.head(5)["domain_pair"].tolist()

print("\nTop 5 domain pairs:")
print(pair_counts.head(5))

pair_counts.to_csv("domain_pair_total_counts.csv", index=False)

# Yearly trend for top 5 pairs
trend = (
    df_pairs[df_pairs["domain_pair"].isin(top5_pairs)]
    .groupby(["year", "domain_pair"])
    .size()
    .reset_index(name="count")
)

# Fill missing years with 0
all_years = sorted(df_pairs["year"].unique())

full_index = pd.MultiIndex.from_product(
    [all_years, top5_pairs],
    names=["year", "domain_pair"]
)

trend_full = (
    trend.set_index(["year", "domain_pair"])
    .reindex(full_index, fill_value=0)
    .reset_index()
)

trend_full.to_csv("top5_domain_pair_yearly_trend.csv", index=False)

print("\nYearly trend:")
print(trend_full)

# Plot
plt.figure(figsize=(8, 4.5))

for pair in top5_pairs:
    temp = trend_full[trend_full["domain_pair"] == pair]
    plt.plot(
        temp["year"],
        temp["count"],
        marker="o",
        linewidth=2,
        label=pair
    )

plt.xlabel("Year")
plt.ylabel("Number of Papers")
plt.title("Temporal Trends of Top Domain Pair Co-occurrences")
plt.grid(True, linestyle="--", alpha=0.5)
plt.legend(fontsize=7)
plt.tight_layout()

plt.savefig("top5_domain_pair_trend.png", dpi=300)
plt.show()

category and subcategory

In [ ]:
import json
import pandas as pd

file_path = "manual_baseline.jsonl"

def safe_json(x):
    if isinstance(x, (dict, list)):
        return x
    if isinstance(x, str):
        try:
            return json.loads(x)
        except json.JSONDecodeError:
            return x
    return None

def normalize_category_field(category_field):
    category_field = safe_json(category_field)

    if isinstance(category_field, dict):
        category_field = category_field.get("datasets", [])

    if not isinstance(category_field, list):
        return []

    clean = []
    for c in category_field:
        c = safe_json(c)
        if isinstance(c, dict):
            clean.append({
                "dataset_name": str(c.get("dataset_name", "")).strip(),
                "category": c.get("category", "unknown"),
                "subcategory": c.get("subcategory", "unknown")
            })

    return clean

rows = []
bad_lines = 0

with open(file_path, "r", encoding="utf-8") as f:
    for i, line in enumerate(f, start=1):
        try:
            paper = json.loads(line.strip())
        except json.JSONDecodeError:
            bad_lines += 1
            continue

        dataset_field = safe_json(paper.get("dataset_name"))
        if not isinstance(dataset_field, dict):
            continue

        datasets = dataset_field.get("datasets", [])
        category_items = normalize_category_field(paper.get("dataset_categories"))

        category_lookup = {}
        for c in category_items:
            if c["dataset_name"]:
                category_lookup[c["dataset_name"]] = {
                    "category": c["category"],
                    "subcategory": c["subcategory"]
                }

        for d in datasets:
            dataset_name = str(d.get("dataset_name", "")).strip()
            action = str(d.get("action", "")).lower().strip()

            if action not in ["created", "used"]:
                continue

            category_info = category_lookup.get(dataset_name, {
                "category": "unknown",
                "subcategory": "unknown"
            })

            rows.append({
                "dataset_name": dataset_name,
                "category": category_info["category"],
                "subcategory": category_info["subcategory"],
                "action": action
            })

df = pd.DataFrame(rows)

print("Invalid lines:", bad_lines)
print("Dataset instances collected:", len(df))
print("\nAction counts:")
print(df["action"].value_counts())

table = (
    df.groupby(["category", "subcategory"])
      .agg(
          Created=("action", lambda x: (x == "created").sum()),
          Used=("action", lambda x: (x == "used").sum())
      )
      .reset_index()
)

table["Total"] = table["Created"] + table["Used"]
table = table.sort_values(by=["category", "Total"], ascending=[True, False])

table.to_csv("category_distribution.csv", index=False)

print("\nFinal table:")
print(table)

In [ ]:
import json
import pandas as pd

file_path = "manual_baseline.jsonl"

def safe_json(x):
    if isinstance(x, (dict, list)):
        return x
    if isinstance(x, str):
        try:
            return json.loads(x)
        except json.JSONDecodeError:
            return x
    return None

def parse_list_field(x):
    x = safe_json(x)

    if isinstance(x, list):
        return sorted(set(str(v).strip() for v in x if str(v).strip()))

    if isinstance(x, str):
        return sorted(set(v.strip() for v in x.split(",") if v.strip()))

    return []

def normalize_category_field(category_field):
    category_field = safe_json(category_field)

    if isinstance(category_field, dict):
        category_field = category_field.get("datasets", [])

    if not isinstance(category_field, list):
        return []

    clean = []

    for c in category_field:
        c = safe_json(c)

        if isinstance(c, dict):
            clean.append({
                "dataset_name": str(c.get("dataset_name", "")).strip(),
                "category": c.get("category", "Unknown"),
                "subcategory": c.get("subcategory", "Unknown")
            })

    return clean

dataset_rows = []
domain_rows = []
bad_lines = 0

with open(file_path, "r", encoding="utf-8") as f:
    for i, line in enumerate(f, start=1):
        line = line.strip()
        if not line:
            continue

        try:
            paper = json.loads(line)
        except json.JSONDecodeError:
            bad_lines += 1
            continue

        dataset_field = safe_json(paper.get("dataset_name"))

        if not isinstance(dataset_field, dict):
            continue

        datasets = dataset_field.get("datasets", [])
        domains = parse_list_field(paper.get("high_level_domains"))

        category_items = normalize_category_field(paper.get("dataset_categories"))

        # match category/subcategory to each dataset by dataset_name
        cat_lookup = {
            c["dataset_name"]: {
                "category": c["category"],
                "subcategory": c["subcategory"]
            }
            for c in category_items
            if c["dataset_name"]
        }

        for d in datasets:
            dataset_name = str(d.get("dataset_name", "")).strip()
            action = str(d.get("action", "")).lower().strip()

            if action not in ["created", "used"]:
                continue

            cat_info = cat_lookup.get(dataset_name, {
                "category": "Unknown",
                "subcategory": "Unknown"
            })

            category = cat_info["category"]
            subcategory = cat_info["subcategory"]

            # real dataset-instance row
            dataset_rows.append({
                "dataset_name": dataset_name,
                "category": category,
                "subcategory": subcategory,
                "action": action
            })

            # exploded only for domain heatmap
            for domain in domains:
                domain_rows.append({
                    "dataset_name": dataset_name,
                    "category": category,
                    "subcategory": subcategory,
                    "domain": domain,
                    "action": action
                })

df = pd.DataFrame(dataset_rows)
df_domain = pd.DataFrame(domain_rows)

print("Bad JSON lines:", bad_lines)
print("Dataset instances:", len(df))
print("\nAction counts:")
print(df["action"].value_counts())

# ======================================================
# 1. CATEGORY / SUBCATEGORY DISTRIBUTION
# ======================================================

dist_table = (
    df.groupby(["category", "subcategory"])
      .agg(
          Created=("action", lambda x: (x == "created").sum()),
          Existing=("action", lambda x: (x == "used").sum())
      )
      .reset_index()
)

dist_table["Total"] = dist_table["Created"] + dist_table["Existing"]

# ======================================================
# 2. TOP 5 DOMAINS OVERALL BY DATASET-DOMAIN USAGE
# ======================================================

top_domains = (
    df_domain["domain"]
    .value_counts()
    .head(5)
    .index
    .tolist()
)

print("\nTop 5 domains:")
print(top_domains)

# ======================================================
# 3. SUBCATEGORY-LEVEL DOMAIN HEATMAP
#    This is the corrected part:
#    each category/subcategory row gets its own domain %
# ======================================================

subcat_domain_counts = pd.crosstab(
    [df_domain["category"], df_domain["subcategory"]],
    df_domain["domain"]
)

subcat_domain_top = subcat_domain_counts[top_domains].copy()

# Row-wise percentage for each subcategory
subcat_domain_percent = (
    subcat_domain_top.div(subcat_domain_top.sum(axis=1), axis=0) * 100
).round(1)

domain_rename = {
    "Security and privacy": "S&P",
    "Computing methodologies": "CM",
    "Software and its engineering": "SE",
    "Information systems": "IS",
    "Networks": "Net"
}

subcat_domain_percent = subcat_domain_percent.rename(columns=domain_rename)
subcat_domain_percent = subcat_domain_percent.reset_index()

# ======================================================
# 4. MERGE DISTRIBUTION + SUBCATEGORY HEATMAP
# ======================================================

merged_table = dist_table.merge(
    subcat_domain_percent,
    on=["category", "subcategory"],
    how="left"
)

# Sort by category and total
merged_table = merged_table.sort_values(
    by=["category", "Total"],
    ascending=[True, False]
)

# Display category only once per group
merged_table["Category_Display"] = merged_table["category"]
merged_table.loc[merged_table.duplicated("category"), "Category_Display"] = ""

heat_cols = [domain_rename.get(d, d) for d in top_domains]

final_table = merged_table[
    ["Category_Display", "subcategory", "Created", "Existing", "Total"] + heat_cols
].rename(columns={
    "Category_Display": "Category",
    "subcategory": "Subcategory"
})

# ======================================================
# 5. SAVE OUTPUTS
# ======================================================

dist_table.to_csv("category_subcategory_distribution.csv", index=False)
subcat_domain_percent.to_csv("subcategory_top5_domain_heatmap_percent.csv", index=False)
final_table.to_csv("merged_subcategory_distribution_heatmap.csv", index=False)

print("\nSubcategory-level heatmap percentages:")
print(subcat_domain_percent)

print("\nMerged table:")
print(final_table)

SUB CAT VS SUB DOMAINS

In [ ]:
import json
import pandas as pd

file_path = "final_baseline_with_paper_type.normalized.subcategory_fixed.updated.jsonl"

def safe_json(x):
    if isinstance(x, (dict, list)):
        return x
    if isinstance(x, str):
        try:
            return json.loads(x)
        except json.JSONDecodeError:
            return x
    return None

def parse_list_field(x):
    x = safe_json(x)

    if isinstance(x, list):
        return sorted(set(str(v).strip() for v in x if str(v).strip()))

    if isinstance(x, str):
        return sorted(set(v.strip() for v in x.split(",") if v.strip()))

    return []

def normalize_category_field(category_field):
    category_field = safe_json(category_field)

    if isinstance(category_field, dict):
        category_field = category_field.get("datasets", [])

    if not isinstance(category_field, list):
        return []

    clean = []
    for c in category_field:
        c = safe_json(c)
        if isinstance(c, dict):
            clean.append({
                "dataset_name": str(c.get("dataset_name", "")).strip(),
                "category": c.get("category", "Unknown"),
                "subcategory": c.get("subcategory", "Unknown")
            })

    return clean

rows = []
bad_lines = 0

with open(file_path, "r", encoding="utf-8") as f:
    for i, line in enumerate(f, start=1):
        try:
            paper = json.loads(line.strip())
        except json.JSONDecodeError:
            bad_lines += 1
            continue

        dataset_field = safe_json(paper.get("dataset_name"))
        if not isinstance(dataset_field, dict):
            continue

        datasets = dataset_field.get("datasets", [])

        # IMPORTANT: your field is "subdomains"
        subdomains = parse_list_field(paper.get("subdomains"))

        if not subdomains:
            continue

        category_items = normalize_category_field(paper.get("dataset_categories"))

        cat_lookup = {
            c["dataset_name"]: {
                "category": c["category"],
                "subcategory": c["subcategory"]
            }
            for c in category_items
            if c["dataset_name"]
        }

        for d in datasets:
            dataset_name = str(d.get("dataset_name", "")).strip()
            action = str(d.get("action", "")).lower().strip()

            if action not in ["created", "used"]:
                continue

            cat_info = cat_lookup.get(dataset_name, {
                "category": "Unknown",
                "subcategory": "Unknown"
            })

            for sd in subdomains:
                rows.append({
                    "dataset_name": dataset_name,
                    "action": action,
                    "category": cat_info["category"],
                    "subcategory": cat_info["subcategory"],
                    "subdomain": sd
                })

df = pd.DataFrame(rows)

print("Bad JSON lines:", bad_lines)
print("Total dataset-subdomain pairs:", len(df))

if df.empty:
    print("No rows collected. Check whether 'subdomains' exists and is populated.")
else:
    print(df.head())
    print("\nAction counts:")
    print(df["action"].value_counts())

    # Top 15 dataset subcategories by dataset-subdomain pair frequency
    top_subcategories = (
        df["subcategory"]
        .value_counts()
        .head(15)
        .index
        .tolist()
    )

    # Top 15 research subdomains
    top_subdomains = (
        df["subdomain"]
        .value_counts()
        .head(15)
        .index
        .tolist()
    )

    df_filtered = df[
        df["subcategory"].isin(top_subcategories) &
        df["subdomain"].isin(top_subdomains)
    ]

    heatmap_counts = pd.crosstab(
        df_filtered["subcategory"],
        df_filtered["subdomain"]
    )

    # Row-wise normalization
    heatmap_percent = (
        heatmap_counts.div(heatmap_counts.sum(axis=1), axis=0) * 100
    ).round(1)

    heatmap_percent = heatmap_percent.loc[top_subcategories]

    heatmap_counts.to_csv("subcategory_subdomain_heatmap_counts.csv")
    heatmap_percent.to_csv("subcategory_subdomain_heatmap_percent.csv")

    print("\nTop 15 subcategories:")
    print(top_subcategories)

    print("\nTop 15 subdomains:")
    print(top_subdomains)

    print("\nHeatmap counts:")
    print(heatmap_counts)

    print("\nHeatmap row-wise percentages:")
    print(heatmap_percent)

creation reason and source

In [ ]:
import json
import pandas as pd

file_path = "manual_baseline.jsonl"

def safe_json(x):
    if isinstance(x, (dict, list)):
        return x
    if isinstance(x, str):
        try:
            return json.loads(x)
        except json.JSONDecodeError:
            return x
    return None

def normalize_created_reasons(reason_field):
    reason_field = safe_json(reason_field)

    if isinstance(reason_field, dict):
        reason_field = reason_field.get("dataset_creation_reason", [])

    if not isinstance(reason_field, list):
        return {}

    out = {}

    for item in reason_field:
        item = safe_json(item)

        if not isinstance(item, dict):
            continue

        dataset_name = str(item.get("dataset_name", "")).strip()
        reasons = item.get("reasons", [])

        if isinstance(reasons, str):
            reasons = [reasons]

        out[dataset_name] = {
            "reason_labels": [],
            "reason_evidence": []
        }

        for r in reasons:
            r = safe_json(r)

            if isinstance(r, dict):
                label = r.get("reason_label", r.get("label", ""))
                evidence = r.get("evidence_span", r.get("evidence", ""))
            else:
                label = str(r)
                evidence = ""

            if label:
                out[dataset_name]["reason_labels"].append(str(label).strip())

            if evidence:
                out[dataset_name]["reason_evidence"].append(str(evidence).strip())

    return out

def normalize_data_sources(source_field):
    source_field = safe_json(source_field)

    if isinstance(source_field, dict):
        source_field = source_field.get("data_sources", [])

    if not isinstance(source_field, list):
        return {}

    out = {}

    for item in source_field:
        item = safe_json(item)

        if not isinstance(item, dict):
            continue

        dataset_name = str(item.get("dataset_name", "")).strip()
        labels = item.get("data_sources_label", item.get("source_labels", item.get("labels", [])))
        evidence = item.get("evidence_span", item.get("evidence", ""))

        if isinstance(labels, str):
            labels = [labels]

        out[dataset_name] = {
            "source_labels": [str(x).strip() for x in labels if str(x).strip()],
            "source_evidence": str(evidence).strip() if evidence else ""
        }

    return out

rows = []
bad_lines = 0

with open(file_path, "r", encoding="utf-8") as f:
    for i, line in enumerate(f, start=1):
        line = line.strip()
        if not line:
            continue

        try:
            paper = json.loads(line)
        except json.JSONDecodeError:
            bad_lines += 1
            continue

        title = paper.get("title", "")
        conference = paper.get("conference", "")
        year = paper.get("year", "")

        dataset_field = safe_json(paper.get("dataset_name"))

        if not isinstance(dataset_field, dict):
            continue

        datasets = dataset_field.get("datasets", [])

        reason_lookup = normalize_created_reasons(
            paper.get("dataset_creation_reason")
        )

        source_lookup = normalize_data_sources(
            paper.get("data_sources")
        )

        for d in datasets:
            dataset_name = str(d.get("dataset_name", "")).strip()
            action = str(d.get("action", "")).lower().strip()

            if action != "created":
                continue

            reason_info = reason_lookup.get(dataset_name, {
                "reason_labels": [],
                "reason_evidence": []
            })

            source_info = source_lookup.get(dataset_name, {
                "source_labels": [],
                "source_evidence": ""
            })

            rows.append({
                "title": title,
                "conference": conference,
                "year": year,
                "dataset_name": dataset_name,
                "action": action,
                "reason_labels": "; ".join(reason_info["reason_labels"]),
                "reason_evidence": " | ".join(reason_info["reason_evidence"]),
                "data_source_labels": "; ".join(source_info["source_labels"]),
                "data_source_evidence": source_info["source_evidence"]
            })

df_created = pd.DataFrame(rows)

print("Bad JSON lines:", bad_lines)
print("Created dataset rows:", len(df_created))

df_created.to_csv("created_datasets_reasons_sources.csv", index=False)

print(df_created.head(20))